Integração com LLM para a Interpretação de Diagnósticos — Prompt Engineering com LLaMA 3.1

O objetivo aqui é demonstrar a integração com LLM local (Ollama + LLaMA 3.1) para gerar explicações em linguagem natural dos diagnósticos produzidos pelos modelos de Machine Learning otimizados pelo Algoritmo Genético.

Técnicas de prompt engineering utilizadas:
- Role prompting (system prompt com papel e restrições);
- Structured input (dados formatados de forma clara e consistente);
- Output format specification (instrução do formato esperado);
- Chain-of-thought (insights acionáveis com raciocínio passo a passo);
- Self-evaluation prompting (avaliação de qualidade da própria resposta).

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import shap

from src.logging_monitor import setup_logging
from src.llm_interpreter import (
    DiagnosisInput, interpret_diagnosis, DEFAULT_MODEL
)

setup_logging()
print("Módulos carregados com sucesso.")

Módulos carregados com sucesso.


1. Preparando os dados e o modelo otimizado.

Carreguei o dataset e instanciei a Regressão Logística com os hiperparâmetros otimizados pelo AG (C=0.131, solver=liblinear) — o modelo com melhor recall após otimização, conforme mostrei no notebook 01.


In [2]:
# Carregar dataset
dataset = fetch_ucirepo(id=17)
X = dataset.data.features
y = (dataset.data.targets.iloc[:, 0] == 'M').astype(int)

# Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Padronização 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Modelo otimizado pelo AG (melhores hiperparâmetros encontrados)
modelo_otimizado = LogisticRegression(
    C=0.131,
    solver='liblinear',
    max_iter=1000,
    random_state=42
)
modelo_otimizado.fit(X_train_scaled, y_train)

print(f"Modelo treinado.")
print(f"Amostras de teste disponíveis: {X_test_scaled.shape[0]}")
print(f"Malignos no teste: {y_test.sum()} | Benignos: {(1-y_test).sum()}")

Modelo treinado.
Amostras de teste disponíveis: 114
Malignos no teste: 42 | Benignos: 72


2. Calculando valores SHAP para o conjunto de teste.

Estou usando os valores SHAP para identificar as features mais relevantes para cada predição individual. Com isso, essas informações serão passadas para a LLM para melhorar as explicações geradas.

In [3]:
masker = shap.maskers.Independent(X_train_scaled, max_samples=455)
explainer = shap.LinearExplainer(modelo_otimizado, masker)
shap_values = explainer(X_test_scaled)

print(f"Valores SHAP calculados: shape={shap_values.values.shape}")
print("Pronto para gerar interpretações.")

Valores SHAP calculados: shape=(114, 30)
Pronto para gerar interpretações.


3. Selecionando casos para interpretação.

Selecionei 3 casos representativos do conjunto de teste:
- Caso A: diagnóstico Maligno com alta confiança.
- Caso B: diagnóstico Benigno com alta confiança.
- Caso C: caso limítrofe (predição próxima de 50%).

In [4]:
# Obter predições e probabilidades
y_pred = modelo_otimizado.predict(X_test_scaled)
y_proba = modelo_otimizado.predict_proba(X_test_scaled)[:, 1]  # prob. de Maligno

feature_names = list(X.columns)

def get_top_features(idx, n=5):
    """Retorna as n features com maior |SHAP value| para o índice dado."""
    shap_vals = shap_values.values[idx]
    patient_vals = X_test_scaled[idx] if isinstance(X_test_scaled, np.ndarray) else X_test_scaled.iloc[idx].values
    top_idx = np.argsort(np.abs(shap_vals))[::-1][:n]
    return [
        {
            "feature": feature_names[i],
            "shap_value": float(shap_vals[i]),
            "patient_value": float(patient_vals[i]),
        }
        for i in top_idx
    ]

# Caso A: Maligno com maior probabilidade
malignos_idx = np.where((y_pred == 1) & (y_test.values == 1))[0]
caso_a_idx = malignos_idx[np.argmax(y_proba[malignos_idx])]

# Caso B: Benigno com menor probabilidade de ser Maligno
benignos_idx = np.where((y_pred == 0) & (y_test.values == 0))[0]
caso_b_idx = benignos_idx[np.argmin(y_proba[benignos_idx])]

# Caso C: caso mais próximo de 0.5 (mais incerto)
caso_c_idx = np.argmin(np.abs(y_proba - 0.5))

casos = {
    "A": caso_a_idx,
    "B": caso_b_idx,
    "C": caso_c_idx,
}

for nome, idx in casos.items():
    pred = "Maligno" if y_pred[idx] == 1 else "Benigno"
    real = "Maligno" if y_test.values[idx] == 1 else "Benigno"
    print(f"Caso {nome} (idx={idx}): predição={pred} | real={real} | prob. maligno={y_proba[idx]:.3f}")

Caso A (idx=108): predição=Maligno | real=Maligno | prob. maligno=1.000
Caso B (idx=91): predição=Benigno | real=Benigno | prob. maligno=0.000
Caso C (idx=112): predição=Benigno | real=Maligno | prob. maligno=0.471


4. Gerando interpretações com a LLM

Caso A — Diagnóstico Maligno:

In [5]:
idx = casos["A"]
diagnosis_a = DiagnosisInput(
    patient_id=f"PACIENTE-TEST-{idx:03d}",
    prediction="Maligno" if y_pred[idx] == 1 else "Benigno",
    confidence=0.9577,  # recall do modelo otimizado
    model_name="Regressão Logística (otimizada por AG)",
    top_features=get_top_features(idx),
    optimized_by_ag=True,
    best_hyperparams={"C": 0.131, "solver": "liblinear"},
)

result_a = interpret_diagnosis(diagnosis_a, model=DEFAULT_MODEL, evaluate=True)

print("=" * 60)
print("EXPLICAÇÃO GERADA:")
print("=" * 60)
print(result_a.explanation)
print("\n" + "=" * 60)
print("INSIGHTS ACIONÁVEIS:")
print("=" * 60)
print(result_a.actionable_insights)
print(f"\nTempo de geração: {result_a.elapsed_seconds}s")
print(f"Qualidade (auto-avaliação): {result_a.quality_score}/10")
print(f"Justificativa: {result_a.quality_justification}")

2026-08-25 17:11:12 | INFO     | src.llm_interpreter | [LLM] Gerando explicação | paciente=PACIENTE-TEST-108 | modelo=llama3.1
2026-08-25 17:15:33 | INFO     | httpx | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-08-25 17:19:02 | INFO     | httpx | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-08-25 17:19:02 | INFO     | src.llm_interpreter | [LLM] Explicação gerada em 469.71s
2026-08-25 17:19:02 | INFO     | src.llm_interpreter | [LLM] Avaliando qualidade | paciente=PACIENTE-TEST-108
2026-08-25 17:21:20 | INFO     | httpx | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-08-25 17:21:20 | INFO     | src.llm_interpreter | [LLM] Qualidade: média=8.5 | scores={'precisao_tecnica': 9, 'clareza': 8, 'seguranca': 9, 'utilidade_clinica': 8}
EXPLICAÇÃO GERADA:
**Resumo do resultado**

O modelo de Regressão Logística otimizado por Algoritmo Genético predisse que a paciente PACIENTE-TEST-108 tem câncer de mama com 

Caso B — Diagnóstico Benigno

In [6]:
idx = casos["B"]
diagnosis_b = DiagnosisInput(
    patient_id=f"PACIENTE-TEST-{idx:03d}",
    prediction="Maligno" if y_pred[idx] == 1 else "Benigno",
    confidence=0.9577,
    model_name="Regressão Logística (otimizada por AG)",
    top_features=get_top_features(idx),
    optimized_by_ag=True,
    best_hyperparams={"C": 0.131, "solver": "liblinear"},
)

print("Gerando interpretação para o Caso B...")
result_b = interpret_diagnosis(diagnosis_b, model=DEFAULT_MODEL, evaluate=True)

print("=" * 60)
print("EXPLICAÇÃO GERADA:")
print("=" * 60)
print(result_b.explanation)
print(f"\nQualidade (auto-avaliação): {result_b.quality_score}/10")

Gerando interpretação para o Caso B...
2026-08-25 17:21:21 | INFO     | src.llm_interpreter | [LLM] Gerando explicação | paciente=PACIENTE-TEST-091 | modelo=llama3.1
2026-08-25 17:25:18 | INFO     | httpx | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-08-25 17:28:17 | INFO     | httpx | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-08-25 17:28:17 | INFO     | src.llm_interpreter | [LLM] Explicação gerada em 416.52s
2026-08-25 17:28:17 | INFO     | src.llm_interpreter | [LLM] Avaliando qualidade | paciente=PACIENTE-TEST-091
2026-08-25 17:30:38 | INFO     | httpx | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-08-25 17:30:38 | INFO     | src.llm_interpreter | [LLM] Qualidade: média=8.5 | scores={'precisao_tecnica': 9, 'clareza': 8, 'seguranca': 9, 'utilidade_clinica': 8}
EXPLICAÇÃO GERADA:
**Resumo do Resultado**

O modelo de diagnóstico assistido por IA predisse que a lesão mamária da paciente PACIENTE-

Caso C — Caso Limítrofe (maior incerteza)

In [7]:
idx = casos["C"]
diagnosis_c = DiagnosisInput(
    patient_id=f"PACIENTE-TEST-{idx:03d}",
    prediction="Maligno" if y_pred[idx] == 1 else "Benigno",
    confidence=0.9577,
    model_name="Regressão Logística (otimizada por AG)",
    top_features=get_top_features(idx),
    optimized_by_ag=True,
    best_hyperparams={"C": 0.131, "solver": "liblinear"},
)

print("Gerando interpretação para o Caso C (limítrofe)...")
result_c = interpret_diagnosis(diagnosis_c, model=DEFAULT_MODEL, evaluate=True)

print("=" * 60)
print("EXPLICAÇÃO GERADA:")
print("=" * 60)
print(result_c.explanation)
print(f"\nQualidade (auto-avaliação): {result_c.quality_score}/10")

Gerando interpretação para o Caso C (limítrofe)...
2026-08-25 17:30:38 | INFO     | src.llm_interpreter | [LLM] Gerando explicação | paciente=PACIENTE-TEST-112 | modelo=llama3.1
2026-08-25 17:34:06 | INFO     | httpx | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-08-25 17:36:49 | INFO     | httpx | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-08-25 17:36:49 | INFO     | src.llm_interpreter | [LLM] Explicação gerada em 370.92s
2026-08-25 17:36:49 | INFO     | src.llm_interpreter | [LLM] Avaliando qualidade | paciente=PACIENTE-TEST-112
2026-08-25 17:39:00 | INFO     | httpx | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-08-25 17:39:00 | INFO     | src.llm_interpreter | [LLM] Qualidade: média=8.5 | scores={'precisao_tecnica': 9, 'clareza': 8, 'seguranca': 9, 'utilidade_clinica': 8}
EXPLICAÇÃO GERADA:
**Resumo do resultado**

O modelo de Regressão Logística otimizado por Algoritmo Genético predisse que a

5. Avaliação comparativa das interpretações
Tabela comparativa de qualidade

In [11]:
rows = []
for nome, result, diag in [
    ("A — Maligno", result_a, diagnosis_a),
    ("B — Benigno", result_b, diagnosis_b),
    ("C — Limítrofe", result_c, diagnosis_c),
]:
    rows.append({
        "Caso": nome,
        "Predição": diag.prediction,
        "Prob. Maligno": f"{y_proba[casos[nome[0]]]:.3f}",
        "Qualidade LLM": f"{result.quality_score:.1f}/10" if result.quality_score else "N/A",
        "Tempo (s)": result.elapsed_seconds,
        "Features top-1": diag.top_features[0]["feature"] if diag.top_features else "N/A",
    })

pd.DataFrame(rows)


,Caso,Predição,Prob. Maligno,Qualidade LLM,Tempo (s),Features top-1
0,A — Maligno,Maligno,1.000,8.5/10,469.71,area3
1,B — Benigno,Benigno,0.000,8.5/10,416.52,fractal_dimension2
2,C — Limítrofe,Benigno,0.471,8.5/10,370.92,texture3


6. Demonstração: pergunta em linguagem natural sobre as rotas

O sistema deve permitir que o médico faça perguntas livres em linguagem natural sobre o diagnóstico.

In [9]:
import ollama

pergunta = """Com base nos resultados apresentados, qual seria a recomendação de acompanhamento para uma paciente com diagnóstico maligno e alta confiança do modelo? Quais exames complementares
seriam indicados?"""

resposta = ollama.chat(
    model=DEFAULT_MODEL,
    messages=[
        {"role": "system", "content": "Você é um assistente médico especializado em oncologia mamária. Responda em português de forma clara e objetiva, sempre reforçando que suas sugestões são de apoio e o médico tem a palavra final."},
        {"role": "user", "content": pergunta},
    ],
    options={"temperature": 0.3},
)

print("PERGUNTA:")
print(pergunta)
print("\nRESPOSTA DA LLM:")
print(resposta["message"]["content"])

2026-08-25 17:41:41 | INFO     | httpx | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
PERGUNTA:
Com base nos resultados apresentados, qual seria a recomendação de acompanhamento para uma paciente com diagnóstico maligno e alta confiança do modelo? Quais exames complementares
seriam indicados?

RESPOSTA DA LLM:
Lamento, mas não posso fornecer recomendações médicas específicas. No entanto, posso oferecer uma visão geral geral sobre o que pode ser considerado em um plano de acompanhamento para uma paciente com diagnóstico de câncer de mama e alta confiança do modelo.

**Plano de Acompanamento:**

1. **Revisão e atualização do plano de tratamento**: É fundamental que o médico revise e atualize o plano de tratamento com base nos resultados do modelo e em outros fatores clínicos relevantes.
2. **Monitoramento da evolução do câncer**: É importante monitorar a evolução do câncer, incluindo a progressão da doença, a resposta ao tratamento e a aparência de novos sintomas.